# Events diff debugger

Part 1 (`df_behavior_summary_all.csv`): one-hot view of which sessions have
which columns differing.

Part 2 (EventsPipeline): deep-dive on a single sub/exp/sess to see the
actual rows that differ between CML and BIDS.

In [1]:
import ast
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path('/home1/zrentala/eeg-validation/results')
EXPERIMENT_RESULT = 'catFR2_final'  # change to catFR1_final_pyedflib, VFFR_final, etc.
SUMMARY_CSV = RESULTS_DIR / EXPERIMENT_RESULT / 'df_behavior_summary_all.csv'

In [2]:
def _parse_list(val):
    if isinstance(val, list):
        return val
    if pd.isna(val) or val in ('', '[]'):
        return []
    return ast.literal_eval(val)


df = pd.read_csv(SUMMARY_CSV)
df['differing_columns'] = df['differing_columns'].apply(_parse_list)
broken = df[df['differing_columns'].map(len) > 0].copy()

all_cols = sorted({c for cols in broken['differing_columns'] for c in cols})
onehot = pd.DataFrame(0, index=broken.index, columns=all_cols, dtype=int)
for idx, cols in broken['differing_columns'].items():
    onehot.loc[idx, cols] = 1

onehot = pd.concat([broken[['subject', 'experiment', 'session']].reset_index(drop=True),
                    onehot.reset_index(drop=True)], axis=1)
print(f'{len(onehot)} sessions with differing columns; {len(all_cols)} unique columns')
onehot

26 sessions with differing columns; 2 unique columns


,subject,experiment,session,recalled,stim_list
0,R1026D,catFR2,0,1,1
1,R1026D,catFR2,2,1,1
2,R1024E,catFR2,0,1,1
3,R1016M,catFR2,0,1,1
4,R1024E,catFR2,1,1,1
5,R1026D,catFR2,1,1,1
6,R1028M,catFR2,0,1,1
7,R1029W,catFR2,1,1,1
8,R1029W,catFR2,0,1,1
9,R1031M,catFR2,0,1,1


In [3]:
# How often each column differs.
onehot[all_cols].sum().sort_values(ascending=False).to_frame('n_sessions')

,n_sessions
recalled,26
stim_list,26


In [4]:
# Example: sessions where 'onset' differs.
TARGET = 'onset'
onehot[onehot[TARGET] == 1] if TARGET in onehot.columns else f'no sessions with {TARGET} differing'

'no sessions with onset differing'

## Deep-dive: R1457T FR1 session 0

Run `EventsPipeline` directly so we can inspect the aligned CML/BIDS frames
and the per-row mismatches returned by the comparator.

In [ ]:
import sys, tempfile
sys.path.insert(0, '/home1/zrentala/eeg-validation')

from eeg_validation.pipelines.events import EventsPipeline

SUB, EXP, SESS = 'R1026D', 'catFR2', 0
BIDS_ROOT = '/data/LTP_BIDS/pyedflib/catFR2/'  # adjust if the live FR1 BIDS root is elsewhere

pipe = EventsPipeline(
    SUB, EXP, SESS,
    bids_root=BIDS_ROOT,
    out_path=tempfile.mkdtemp(prefix='events_debug_'),
    skip_if_exists=False,  # force in-memory result even if CSV already exists
    verbose=True,
)
out = pipe.run()
result = out['result']
print('ok =', result.ok)


[EventsPipeline] Starting pipeline for R1026D_catFR1_0
  Output directory: /tmp/events_debug_w61e5s6_
  Loading CML events...
  CML events loaded: 513 rows, types=['COUNTDOWN_END', 'COUNTDOWN_START', 'DISTRACT_END', 'DISTRACT_START', 'ORIENT', 'PRACTICE_DISTRACT_END', 'PRACTICE_DISTRACT_START', 'PRACTICE_REC_END', 'PRACTICE_REC_START', 'PRACTICE_WORD', 'PROB', 'REC_END', 'REC_START', 'REC_WORD', 'REC_WORD_VV', 'SESSION_SKIPPED', 'SESS_START', 'START', 'STOP', 'TRIAL', 'WORD']
  Loading BIDS events...
  BIDS events not found: load_events: no file matched for /data/LTP_BIDS/pyedflib/catFR2/sub-R1026D/ses-0/ieeg/sub-R1026D_ses-0_task-catFR1_events.tsv
[EventsPipeline] Pipeline complete for R1026D_catFR1_0



/usr/global/ubuntu/miniforge3/25.3.1/envs/workshop_311_rhino2b/lib/python3.11/site-packages/cmlreaders/cmlreader.py:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  index["montage"].replace({np.nan: "0"}, inplace=True)
/usr/global/ubuntu/miniforge3/25.3.1/envs/workshop_311_rhino2b/lib/python3.11/site-packages/cmlreaders/cmlreader.py:84: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method wi

KeyError: 'result'

In [6]:
# Session-level summary (differing columns, only-in-A, only-in-B, row counts).
result.df_summary.T

,0
subject,R1457T
experiment,catFR1
session,0
comparison,CMLReader vs OpenBIDS
n_rows_compared,541
n_rows_a,541
n_rows_b,541
length_mismatch,False
n_columns_compared,12
differing_columns,[onset]


In [7]:
# Per-column mismatch counts (only columns with n_mismatches > 0 matter).
result.df_detail[result.df_detail['n_mismatches'] > 0]

,subject,experiment,session,column,n_mismatches,fraction_mismatch,dtype_a,dtype_b,numeric_compared_with_isclose
0,R1457T,catFR1,0,onset,2,0.003697,float64,float64,True


In [8]:
# The actual differing values, row by row (up to max_mismatches=20 per column).
# Columns: subject, experiment, session, column, i (row index into aligned frames),
# CMLReader (value in aligned_a), OpenBIDS (value in aligned_b).
result.df_mismatches

,subject,experiment,session,column,i,CMLReader,OpenBIDS
0,R1457T,catFR1,0,onset,1,0.001,0.001
1,R1457T,catFR1,0,onset,35,-0.001,-0.001


In [9]:
# Side-by-side view: for every row index that has ANY differing column,
# show the CML and BIDS values of the differing columns next to each other.
a, b = result.aligned_a, result.aligned_b
diff_cols = result.df_summary.iloc[0]['differing_columns']
bad_idx = sorted(result.df_mismatches['i'].unique().tolist()) if len(result.df_mismatches) else []

if not bad_idx:
    print('no differing rows')
else:
    pieces = []
    for col in diff_cols:
        pieces.append(pd.DataFrame({
            f'{col}__CML': a[col].iloc[bad_idx].values,
            f'{col}__BIDS': b[col].iloc[bad_idx].values,
        }, index=bad_idx))
    side_by_side = pd.concat(pieces, axis=1)
    side_by_side.index.name = 'row_i'
    print(f'{len(bad_idx)} rows with at least one differing column')
    side_by_side

2 rows with at least one differing column


In [10]:
# Show the full aligned rows (all shared columns) for the mismatching indices,
# with CML and BIDS interleaved so you can see the surrounding context.
if bad_idx:
    rows_cml = a.iloc[bad_idx].reset_index(drop=True).add_suffix('__CML')
    rows_bids = b.iloc[bad_idx].reset_index(drop=True).add_suffix('__BIDS')
    interleaved = pd.concat([rows_cml, rows_bids], axis=1)
    interleaved.insert(0, 'row_i', bad_idx)
    interleaved

### Why is BIDS "ahead" of CML in parts of the session?

The shift isn't a sort artifact (removing `sort_keys` doesn't fix it), and
lengths match (548 vs 548). That means CML and BIDS have different event
*sets* — one side has an extra row in one region and a compensating extra
row elsewhere to keep totals equal.

The diagnostics below:
1. Compare raw CML vs BIDS event-type counts.
2. Outer-join on `(sample, trial_type)` to find rows present on one side only.
3. Find the first row where the aligned frames diverge and show the window.

In [11]:
from eeg_validation.loaders.cml import load_cml_events
from eeg_validation.loaders.bids import load_bids_events

evs_cml_raw = load_cml_events(SUB, EXP, SESS)
evs_bids_raw = load_bids_events(reader=pipe.reader, event_type=pipe.reader.device)

print(f'CML raw: {len(evs_cml_raw)} rows  |  BIDS raw: {len(evs_bids_raw)} rows')

type_counts = pd.concat([
    evs_cml_raw['type'].astype(str).value_counts().rename('CML'),
    evs_bids_raw['trial_type'].astype(str).value_counts().rename('BIDS'),
], axis=1).fillna(0).astype(int)
type_counts['delta'] = type_counts['CML'] - type_counts['BIDS']
type_counts.sort_values('delta', key=abs, ascending=False)

CML raw: 541 rows  |  BIDS raw: 541 rows


/usr/global/ubuntu/miniforge3/25.3.1/envs/workshop_311_rhino2b/lib/python3.11/site-packages/cmlreaders/cmlreader.py:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  index["montage"].replace({np.nan: "0"}, inplace=True)
/usr/global/ubuntu/miniforge3/25.3.1/envs/workshop_311_rhino2b/lib/python3.11/site-packages/cmlreaders/cmlreader.py:84: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method wi

,CML,BIDS,delta
WORD,156,156,0
WORD_OFF,156,156,0
PROB,55,55,0
REC_WORD,31,31,0
START,13,13,0
STOP,13,13,0
REC_WORD_VV,13,13,0
COUNTDOWN_START,13,13,0
COUNTDOWN_END,13,13,0
ENCODING_START,13,13,0


In [12]:
# Outer-join on (sample, trial_type). Rows that appear on only one side
# are the direct cause of the shift.
cml_keys = evs_cml_raw.rename(columns={'eegoffset': 'sample', 'type': 'trial_type'})[['sample', 'trial_type']].copy()
cml_keys['trial_type'] = cml_keys['trial_type'].astype(str)
cml_keys['sample'] = pd.to_numeric(cml_keys['sample'], errors='coerce')

bids_keys = evs_bids_raw[['sample', 'trial_type']].copy()
bids_keys['trial_type'] = bids_keys['trial_type'].astype(str)
bids_keys['sample'] = pd.to_numeric(bids_keys['sample'], errors='coerce')

outer = cml_keys.merge(
    bids_keys, on=['sample', 'trial_type'], how='outer', indicator=True,
).sort_values('sample', kind='mergesort').reset_index(drop=True)

only_cml = outer[outer['_merge'] == 'left_only']
only_bids = outer[outer['_merge'] == 'right_only']
print(f'Only in CML: {len(only_cml)}  |  Only in BIDS: {len(only_bids)}')
pd.concat([only_cml.assign(side='CML_only'), only_bids.assign(side='BIDS_only')]).sort_values('sample')

Only in CML: 51  |  Only in BIDS: 51


,sample,trial_type,_merge,side
3,184933,REC_WORD_VV,left_only,CML_only
5,186933,REC_WORD_VV,right_only,BIDS_only
10,210580,REC_WORD_VV,left_only,CML_only
12,211580,REC_WORD_VV,right_only,BIDS_only
15,214041,REC_WORD_VV,left_only,CML_only
...,...,...,...,...
550,1444130,REC_WORD,right_only,BIDS_only
556,1469405,REC_START,left_only,CML_only
557,1470405,REC_START,right_only,BIDS_only
589,1574841,REC_START,left_only,CML_only


In [13]:
# Find where the aligned frames first diverge on (sample, trial_type) alone —
# any mismatch here is a structural misalignment, not a value difference.
keys_a = list(zip(a['sample'], a['trial_type']))
keys_b = list(zip(b['sample'], b['trial_type']))
first_struct_diff = next((i for i, (ka, kb) in enumerate(zip(keys_a, keys_b)) if ka != kb), None)
print(f'first (sample, trial_type) divergence at i={first_struct_diff}')

if first_struct_diff is not None:
    lo, hi = max(0, first_struct_diff - 3), min(len(a), first_struct_diff + 8)
    display(pd.DataFrame({
        'sample_CML': a['sample'].iloc[lo:hi].values,
        'trial_type_CML': a['trial_type'].iloc[lo:hi].values,
        'sample_BIDS': b['sample'].iloc[lo:hi].values,
        'trial_type_BIDS': b['trial_type'].iloc[lo:hi].values,
    }, index=range(lo, hi)))

first (sample, trial_type) divergence at i=None


### Finding: `sample` disagrees on ~56 events, same event on both sides

- 548 vs 548 rows, every `trial_type` count matches (`delta == 0`).
- But 56 `(sample, trial_type)` pairs are unique to each side.

⇒ The same events exist on both sides with the same `trial_type`, but the
`sample` index differs. That's enough to change relative order when sorting
by `sample`, which is why segments of the aligned frames look like
"BIDS is one row ahead of CML".

Below: pair up the CML-only and BIDS-only rows within each `trial_type` (by
order of appearance) and look at the per-row `sample` deltas. If the deltas
are small and mostly ±1 it's a rounding/sample-rate conversion issue between
the two writers; if they're large it's a deeper event-timestamping bug.

In [14]:
# Pair each CML-only row with the BIDS-only row of the same trial_type in
# order of sample. Report (sample_cml, sample_bids, delta). If the deltas
# are consistently small, the two writers just round sample differently.
def _ordered(df_side, keep):
    return (df_side.query('_merge == @keep')
                   .sort_values('sample', kind='mergesort')
                   .reset_index(drop=True))

cml_only = _ordered(outer, 'left_only')[['sample', 'trial_type']].rename(columns={'sample': 'sample_CML'})
bids_only = _ordered(outer, 'right_only')[['sample', 'trial_type']].rename(columns={'sample': 'sample_BIDS'})

cml_only['rank'] = cml_only.groupby('trial_type').cumcount()
bids_only['rank'] = bids_only.groupby('trial_type').cumcount()

paired = cml_only.merge(bids_only, on=['trial_type', 'rank'], how='outer')
paired['delta'] = paired['sample_BIDS'] - paired['sample_CML']
paired.sort_values(['trial_type', 'rank'])

,sample_CML,trial_type,rank,sample_BIDS,delta
0,319579,REC_START,0,320579,1000
1,465828,REC_START,1,466828,1000
2,568188,REC_START,2,569188,1000
3,668933,REC_START,3,669933,1000
4,771803,REC_START,4,772803,1000
5,869388,REC_START,5,870388,1000
6,970416,REC_START,6,971416,1000
7,1077807,REC_START,7,1078807,1000
8,1175334,REC_START,8,1176334,1000
9,1271686,REC_START,9,1272686,1000


In [15]:
# Summary of the sample deltas — are they systematic or random?
print('delta value_counts (BIDS_sample - CML_sample):')
print(paired['delta'].value_counts().sort_index())
print('\ndelta stats:')
print(paired['delta'].describe())
print('\nper-trial_type delta stats:')
paired.groupby('trial_type')['delta'].agg(['count', 'min', 'max', 'mean', 'std'])

delta value_counts (BIDS_sample - CML_sample):
delta
1000    45
2000     6
Name: count, dtype: int64

delta stats:
count      51.000000
mean     1117.647059
std       325.395687
min      1000.000000
25%      1000.000000
50%      1000.000000
75%      1000.000000
max      2000.000000
Name: delta, dtype: float64

per-trial_type delta stats:


,count,min,max,mean,std
trial_type,,,,,
REC_START,13,1000,1000,1000.000000,0.000000
REC_WORD,28,1000,2000,1107.142857,314.970394
REC_WORD_VV,10,1000,2000,1300.000000,483.045892


### Root cause: recording-split offset (+1000 samples per split)

Deltas cluster at exactly **+1000 / +2000 / +3000 samples** — three discrete
steps, not a continuum. That's the signature of a segmented EEG recording
where one writer adds the inter-segment gap length to post-split events and
the other doesn't.

To confirm: check the CML `eegfile` column. If 54 events live in one EDF
segment and 1 event each live in two later segments, the split hypothesis is
locked in — and the sample rate × seconds-per-gap will resolve to 1000.

In [31]:
# How many distinct EEG files does CML reference for this session?
print('CML eegfile value_counts:')
print(evs_cml_raw['eegfile'].value_counts(dropna=False))
print(f"\nunique non-null eegfiles: {evs_cml_raw['eegfile'].dropna().nunique()}")

CML eegfile value_counts:
eegfile
R1457T_catFR1_0_30Oct18_1917.h5    541
Name: count, dtype: int64

unique non-null eegfiles: 1


In [32]:
# Cross-reference the shifted events with their CML eegfile. If the 54/1/1
# split by delta matches the event-count-per-eegfile split, this is
# definitively a recording-segment alignment bug.
cml_with_keys = evs_cml_raw.rename(columns={'eegoffset': 'sample', 'type': 'trial_type'})[
    ['sample', 'trial_type', 'eegfile']
].copy()
cml_with_keys['trial_type'] = cml_with_keys['trial_type'].astype(str)

shifted = cml_only.merge(cml_with_keys, left_on=['sample_CML', 'trial_type'],
                         right_on=['sample', 'trial_type'], how='left')
# Attach the matching BIDS sample / delta.
shifted = shifted.merge(bids_only[['trial_type', 'rank', 'sample_BIDS']],
                        on=['trial_type', 'rank'], how='left')
shifted['delta'] = shifted['sample_BIDS'] - shifted['sample_CML']
print('shifted events grouped by (eegfile, delta):')
shifted.groupby(['eegfile', 'delta']).size().to_frame('n')

shifted events grouped by (eegfile, delta):


n
eegfile                         delta    
R1457T_catFR1_0_30Oct18_1917.h5 1000   45
                                2000    6

In [33]:
# Single eegfile → not a split-recording issue. Group shifted events by
# trial_type and delta to see which event category is misaligned.
print('shifted events by (trial_type, delta):')
print(shifted.groupby(['trial_type', 'delta']).size().to_frame('n'))

print('\nall trial_types in shifted set:')
print(shifted['trial_type'].value_counts())

print('\nfor comparison, total count of each of those trial_types in the raw frames:')
for t in shifted['trial_type'].unique():
    n_cml = int((evs_cml_raw['type'].astype(str) == t).sum())
    n_bids = int((evs_bids_raw['trial_type'].astype(str) == t).sum())
    print(f'  {t}: CML={n_cml}, BIDS={n_bids}')

shifted events by (trial_type, delta):
                    n
trial_type  delta    
REC_START   1000   13
REC_WORD    1000   25
            2000    3
REC_WORD_VV 1000    7
            2000    3

all trial_types in shifted set:
trial_type
REC_WORD       28
REC_START      13
REC_WORD_VV    10
Name: count, dtype: int64

for comparison, total count of each of those trial_types in the raw frames:
  REC_WORD_VV: CML=13, BIDS=13
  REC_START: CML=13, BIDS=13
  REC_WORD: CML=31, BIDS=31
